In [1]:
AttachSpec("finite-flat-covers/spec");
SetColumns(100);

// And helper functions
function StrFact(f)
  fct := Factorisation(f);
  ret := "";
  for g in fct do
    ret cat:= Sprintf("(%o)^(%o) * ", g[1], g[2]);
  end for;
  return ret[1..(#ret - 3)];
end function;

function PrettyPrint(A)
  m := Nrows(A);
  n := Ncols(A);
  // Compute the width of each column
  widths := [
    Max([#Sprint(A[i,j]) : i in [1..j]])
    : j in [1..n]
  ];
  ret := "";
  for i in [1..m] do
    ret cat:= "[";
    for j in [1..n] do
      if j ge i then
        ret cat:= Sprintf(" %*o", widths[j], A[i,j]);
      else
        ret cat:= Sprintf(" %*o", widths[j], "");
      end if;
    end for;
    ret cat:= " ]\n";
  end for;
  return ret;
end function;

function PrintSyz(A : width:=80)
  P := BaseRing(A);
  R := BaseRing(P);

  gap := "    ";

  blocks := [];
  widths := [];

  for i in [1..Rank(P)] do
    B := Matrix(R, Nrows(A), Ncols(A),
      [MonomialCoefficient(A[r,c], P.i) :
       r in [1..Nrows(A)], c in [1..Ncols(A)]]);
    lines := [Sprintf("A_%o =", i)] cat Split(PrettyPrint(B));
    Append(~blocks, lines);
    Append(~widths, Max([#s : s in lines]));
  end for;

  N := #blocks;
  if N eq 0 then
    return "";
  end if;

  error if Max(widths) gt width,
    "A matrix is wider than the specified width";

  cols := 1;
  colWidths := [Max(widths)];
  for k in [1..N] do
    candidate := [
      Max([widths[i] : i in [1..N]
       | (i-1) mod k eq j-1])
       : j in [1..k]
      ];

    total := &+candidate + (k-1)*#gap;

    if total le width then
      cols := k;
      colWidths := candidate;
    end if;
  end for;

  ret := "";
  start := 1;

  while start le N do
    last := Min(start + cols - 1, N);
    height := Max([#blocks[i] : i in [start..last]]);
    for r in [1..height] do
      for j in [1..last-start+1] do
        i := start + j - 1;
        s := "";
        if r le #blocks[i] then
          s := blocks[i][r];
        end if;

        if i lt last then
          ret cat:= Sprintf("%*o", -colWidths[j], s);
          ret cat:= gap;
        else
          ret cat:= s;
        end if;
      end for;

      ret cat:= "\n";
    end for;

    if last lt N then
      ret cat:= "\n";
    end if;

    start +:= cols;
  end while;

  return ret;
end function;

<style>
:root {
    --r-main-font-size: 36px;
    --jp-content-font-size1: 28px;
    --jp-code-font-size: 24px;
}

.reveal .jp-RenderedHTMLCommon {
    font-size: 28px !important;
}

.reveal blockquote {
    width: 100% !important;
    margin-left: 0 !important;
    margin-right: 0 !important;
    padding-left: 0 !important;
    text-indent: 0 !important;
    text-align: left;
    box-sizing: border-box;
}
</style>

# Tschirnhausen bundles of quintic covers of $\mathbb{P}^1$

### Sam Frengley (joint with Sameera Vemulapalli)

https://github.com/SamFrengley/quintic-covers-lleida.git


This is a Jupyter notebook (Magma needed). To play along at home you can clone 

https://github.com/SamFrengley/quintic-covers-lleida.git

(It is also linked on my webpage.)

## Quintic fields 

### An example of a quintic field

Suppose I have some degree $5$ polynomials with coefficients in $\mathbb{Q}$, say 

In [2]:
ZZ := Integers();
P<x> := PolynomialRing(ZZ);
f := x^5 + 3*x^3 + 2*x + 1;
g := 537824*x^5 + 806736*x^4 + 5306896*x^3 - 7141064*x^2 
     + 4296558*x - 358787;

In [3]:
printf "disc(f) = %o\n\n", StrFact(Discriminant(f));
printf "disc(g) = %o\n", StrFact(Discriminant(g));

disc(f) = (59)^(1) * (83)^(1)

disc(g) = (2)^(40) * (7)^(20) * (59)^(1) * (71)^(2) * (83)^(1) * (167)^(2) * (2463473)^(2)


If we know them in advance, we can check they define the same field

In [4]:
K := NumberField(f); // The field ℚ[x]/f(x)
L := NumberField(g); // The field ℚ[x]/g(x)
flag, _ := IsIsomorphic(K, L);

In [5]:
printf "These fields are isomorphic: flag = %o", flag;

These fields are isomorphic: flag = true


But what if I just start with the horrible polynomial $g(x)$, **how do we find the "very nice" polynomial $f(x)$**? 

### First try: Mobius transformations

We could try Mobius transformations by setting $x = \tfrac{ay + b}{cy + d}$. This is implemented in Magma: the very best we can do this way is:

In [6]:
// Apply Mobius maps until g is "as nice as possible" 
g_2 := MinRedBinaryForm(g);

In [7]:
printf "g_2 = %o", g_2;

g_2 = x^5 - 23*x^4 + 651*x^3 + 378*x^2 + 6327*x - 16339


But that's nowhere near as nice as $f(x) = x^5 + 3x^3 + 2x + 1$.

### Better try: Maximal orders 

A better way would be to compute the maximal order in $L$, and then take a short basis vector:

In [8]:
O_L := MaximalOrder(L); // The ring of integers
B := Basis(LLL(O_L)); // A short basis for O_L
new_g := MinimalPolynomial(B[2]);

In [9]:
printf "Our original polynomial %o\n", f;
printf "Our new polynomial %o\n\n", new_g;
printf "disc(f) = disc(new_g) = %o\n", StrFact(Discriminant(new_g));

Our original polynomial x^5 + 3*x^3 + 2*x + 1
Our new polynomial x^5 - x^4 - x^3 + x^2 + 1

disc(f) = disc(new_g) = (59)^(1) * (83)^(1)


In [10]:
// This is essentially what goes on inside (e.g.)
Lprime := OptimisedRepresentation(L);

In [11]:
printf "Lprime = ℚ[x] / (%o)", DefiningPolynomial(Lprime);

Lprime = ℚ[x] / (x^5 - x^4 - x^3 + x^2 + 1)


> **Important takeaway**: Polynomials which define the same field need not be related by Mobius tranformations!

## The geometric analogue

<p align="center">
<img src="ramified_cover.png" width="800">
</p>

### How to write down a degree $5$ cover?

From now on we will write 
$$ \mathbf{e} = (e_1, e_2, e_3, e_4) \qquad \text{and} \qquad \mathbf{f} = (f_1, f_2, f_3, f_4, f_5) $$
where 
$$ 1 \leq e_1 \leq \dots \leq e_4 \qquad \text{and} \qquad 1 \leq f_1 \leq \dots \leq f_5$$ 

Recall that an $(n \times n)$-matrix $A$ (away from characteristic 2) is called *alternating* if $A^t = -A$.

> **Theorem:** (Casnati--Ekhedal, Bhargava) Let $\pi \colon C \to \mathbb{P}^1$ be a cover of degree $5$. Then there exists $\mathbf{e} \in \mathbb{Z}^4$ and $\mathbf{f} \in \mathbb{Z}^5$ and alternating matrices $A_1, A_2, A_3, A_4 \in \mathrm{Mat}_{5 \times 5}(F[s,t])$ such that
> \begin{align*} C \cong \{ ([s : t], [x_1 : x_2 : x_3 : x_4]) : \mathrm{rank}(\mathcal{A}) \leq 2 \} \subset \mathbb{P}( \mathcal{E} )\end{align*}
> where
> $$ \mathcal{A} = x_1 A_1 + x_2 A_2 + x_3 A_3 + x_4 A_4, $$
> and if $A_k = (a_{ij}^{(k)} )_{1 \leq i,j \leq 5}$ then
> $$ \deg(a_{ij}^{(k)}) = f_i + f_j + e_k - (g + 4)$$
> and $g = \sum_{i=1}^4 e_i - 4$ is the (arithmetic) genus of $C$.

### A worked example

Consider the polynomial in $\mathbb{F}_{17}[x,y]$

\begin{align*}
f(x, y) =&\;  t^2 (x^5 - 5 x y^4 + 4 y^5) + 4 s t (x - y) y^4 - s^2 y (x^4 + y^4) \\
\end{align*}

> This defines a curve $C \subset \mathbb{P}^1 \times \mathbb{P}^1$ and the resulting map
> $$ \pi \colon C \to \mathbb{P}^1 $$
> (given by projection onto $[s:t]$) has degree $5$.

> The Casnati--Ekhedahl / Bhargava construction embeds
> $$ C \hookrightarrow \mathbb{P}^1 \times \mathbb{P}^1 \hookrightarrow \mathbb{P}^1 \times \mathbb{P}^3 = \mathbb{P}(\mathcal{E})$$
> as the twisted cubic on the second $\mathbb{P}^1$. 

In [12]:
// Setup
FF := FiniteField(17);
R<s,t> := PolynomialRing(FF, 2); K<S,T> := FieldOfFractions(R);

// The polynomial ring 𝔽_{17}[s,t][x] 
P<x> := PolynomialRing(R); 

// The polynomial f(x)
f := t^2*(x^5 - 5*x + 4) + 4*s*t*(x - 1) - s^2*(x^4 + 1);

In [13]:
// The discriminant
printf "disc(f) = %o * (...)^(1)\n", StrFact(Discriminant(f))[1..40];

disc(f) = (s)^(2) * (s + 4*t)^(2) * (s + 14*t)^(1) * (...)^(1)


In [14]:
// Command which initialises the curve C as the vanishing of 
// the Pfaffians of four 5x5 alternating matrices
A := FiniteFlatCover(f : ProjectiveBase:=true, Twisted:=false);

In [15]:
// The 5x5 matrices
cA := FirstSyzygies(A);
PrintSyz(cA);

A_1 =                                               A_2 =
[ 0 0 0                       0              0 ]    [ 0 0 0 0 1 ]
[   0 0                       0             16 ]    [   0 0 1 0 ]
[     0 16*s^2 + 13*s*t + 4*t^2 4*s*t + 12*t^2 ]    [     0 0 0 ]
[                             0              0 ]    [       0 0 ]
[                                            0 ]    [         0 ]

A_3 =                                               A_4 =
[ 0 0  0 16      0 ]                                [ 0 0 1 0   0 ]
[   0 16  0      0 ]                                [   0 0 0   0 ]
[      0  0      0 ]                                [     0 0   0 ]
[         0 16*s^2 ]                                [       0 t^2 ]
[                0 ]                                [           0 ]



In [16]:
// You can access all the Intrinsics via 
ListSignatures(Type(A))

Signatures relevant to FinFlatCov:

    '->'(S::Str, T::Str) -> SetCart
    ActBy(~C::FinFlatCov, g::Mtrx, h::Mtrx)
    ActBy(C::FinFlatCov, g::Mtrx) -> FinFlatCov
    ActBy(~C::FinFlatCov, g::Mtrx, H::List)
    ActBy(~C::FinFlatCov, g::Mtrx, H::SeqEnum)
    ActBy(C::FinFlatCov, g::Mtrx, H::List) -> FinFlatCov
    ActBy(C::FinFlatCov, g::Mtrx, h::Mtrx) -> FinFlatCov
    ActBy(~C::FinFlatCov, g::Mtrx)
    ActBy(C::FinFlatCov, g::Mtrx, H::SeqEnum) -> FinFlatCov
    AffineAlgebraScheme(C::FinFlatCov) -> Sch
    AlgebraBasisChange(C::FinFlatCov) -> Mtrx
    AmbientBundle(C::FinFlatCov) -> Sch
    AssociativeArray(I::Str) -> Assoc
    Aut(S::Str) -> PowMapAut, Map
    Bang(D::Str, C::Str) -> Map
    BaseChange(C::FinFlatCov, pi::RngMPolElt) -> FinFlatCov, Map
    BaseChange(C::FinFlatCov, phi::Map) -> FinFlatCov, Map
    BaseChange(C::FinFlatCov, p::RngIntElt) -> FinFlatCov, Map
    BaseChange(C::FinFlatCov, pi::RngUPolElt) -> FinFlatCov, Map
    BaseChange(C::FinFlatCov, K::Fld, values::Se

## Legal transformations

We want to act on the set of 4-tuples of alternating matrices $(A_1, A_2, A_3, A_4)$ with 
$$ \deg(a_{ij}^{(k)}) = d_{ij}^{(k)} = f_i + f_j + e_k - (g + 4)$$
while preserving the locus $\mathrm{rank}(\mathcal{A}) \leq 2$ up to isomorphism (where $\mathcal{A} = x_1 A_1 + x_2 A_2 + x_3 A_3 + x_4 A_4$)


#### Transformation 1 

Add 'left' matrices to 'right' matrices e.g., 
$$ A_4 \leftarrow A_4 + (\text{polynomial of degree $e_4 - e_1$}) \cdot A_1 $$ 

> This action is captured by (block) lower triangular $4 \times 4$-matrices 
> $$ \mathrm{GL}(\mathcal{E}) = \left\{ U = (u_{ij}) \in \mathrm{Mat}_{4\times 4}(k[s,t]) : \det(U) \neq 0 \text{ and } \deg(u_{ij}) = e_j - e_i \right\}$$
> via multiplication $U \cdot (A_1, \cdots, A_4)$

#### Transformation 2 

Add 'left' rows/columns to 'right' rows/columns e.g., 
$$ a_{5j}^{(k)} \leftarrow a_{5j}^{(k)} + (\text{polynomial of degree $f_5 - f_1$}) \cdot a_{1j}^{(k)} $$ 

>This action is captured by (block) lower triangular $5 \times 5$-matrices 
> $$ \mathrm{GL}(\mathcal{F}) = \left\{ V = (v_{ij}) \in \mathrm{Mat}_{5\times 5}(k[s,t]) : \det(V) \neq 0 \text{ and } \deg(v_{ij}) = f_j - f_i \right\}$$
> via $V^t A_k V$

### The algorithm

There are 2 steps:

1. Move the matrices into a "good shape" via legal moves

2. Apply a "desingularisation operation"

## The conditions on singular fibres

> **Theorem**: (Bhargava, F.--Vemulapalli) Let $p \in F[s,t]$ be a linear polynomial. There exists a singular point of $C$ in the fibre of $\pi \colon C \to \mathbb{P}^1$ above $\{p = 0\}$ if and only if $\mathcal{A}$ can be put in one of the following two shapes (algorithmically) **via only legal moves**:
>
> ----------------------
> _Case 1_: $A_4 = 0 \pmod{p}$
>
> ----------------------
> _Case 2_:
> $$ A_1, A_2, A_3 = \begin{pmatrix} 0 & * & * & * & * \\ & 0 & * & * & * \\ && 0 & * & * \\ &&& 0 & * \, p \\ &&&& 0 \end{pmatrix} \quad A_4 = \begin{pmatrix} 0 & * & * & * \, p & * \, p \\ & 0 & * & * \, p & * \, p \\ && 0 & * \, p & * \, p \\ &&& 0 & * \, p^2 \\ &&&& 0 \end{pmatrix} $$

### Desingularisation operation

Case 1:

$$ A_4 \leftarrow A_4 / p $$

-----------------

Case 2:

The $\mathrm{GL}(\mathcal{E})$ transformation (multiply the first 3 matrices by $p$):
$$ g_4 = \mathrm{diag}(p, \; p, \; p, \; 1) $$

The $\mathrm{GL}(\mathcal{F})$ transformation (divide the last 2 rows by $p$):
$$ g_5 = \mathrm{diag}(1, \; 1, \; 1, \; 1/p, \; 1/p) $$


### Back to the example

The curve $C \subset \mathbb{P}^1 \times \mathbb{P}^1$ has a singular point where $s = 0$. Its image under $C \hookrightarrow \mathbb{P}^1 \times \mathbb{P}^3$ is the singular point
$$ ([s:t], [x_1 : x_2 : x_3 : x_4]) = ([0:1], [-1:-1:-1:1]) $$

In [17]:
C := UnderlyingScheme(A);
sing := ReducedSubscheme(SingularSubscheme(C));

In [18]:
printf "The singular subscheme of C is cut out" cat
  " by the polynomials \n%o\n\n", DefiningPolynomials(sing);

The singular subscheme of C is cut out by the polynomials 
[
s,
X[1] + 16*X[4],
X[2] + 16*X[4],
X[3] + 16*X[4]
]



In [19]:
p := s;

// Move the singular point to [0:0:0:1]
g4 := Matrix(K, 4, 4, 
[
  [1, 0, 0, 0],
  [0, 1, 0, 0],
  [0, 0, 1, 0],
  [1, 1, 1, 1]
]);

// 5x5 matrix transformation to put things in 
// the correct format
g5 := Matrix(K, 5, 5, 
[
  [     1,      0,      0,      0,      0],
  [     1,      1,      0,      0,      0],
  [16*t^2,      0,      0,      0,      1],
  [   t^2,  2*t^2,      0,      1,      1],
  [ 2*t^2, 14*t^2,      1,      0,     16]
]);

B := A^<g5,g4>;

In [20]:
print "After making the transformations we get a model with:\n";
PrintSyz(FirstSyzygies(B));

After making the transformations we get a model with:

A_1 =                               A_2 =
[ 0 0 0      0              0 ]     [ 0 0 16    16   1 ]
[   0 1      1             16 ]     [   0 16    15   1 ]
[     0 15*t^2 4*s*t + 15*t^2 ]     [      0 2*t^2 t^2 ]
[            0         16*s^2 ]     [            0   0 ]
[                           0 ]     [                0 ]

A_3 =                               A_4 =
[ 0 0 0               1      0 ]    [ 0 0 16      0      0 ]
[   0 0               1      1 ]    [   0  0      0      0 ]
[     0 16*s^2 + 16*t^2      0 ]    [      0 16*s^2  4*s*t ]
[                     0 16*s^2 ]    [             0 15*s^2 ]
[                            0 ]    [                    0 ]



In [21]:
// First choose a 4x4 matrix g4 which moves the singular point on the special fibre to [0:0:0:1] 
B := A^g4;

// Now we want to choose a 5x5 matrix which puts the model in the special form
g5 := IdentityMatrix(K, 5);

/*
The point is to make the kernel of A_4 base changed to the fibre s=0 (which thus has rank 2) 
in the form
  [ 0 0 1 0 0 ]
  [ 0 0 0 1 0 ]
  [ 0 0 0 0 1 ]
*/ 

// Kernel(Model(BaseChange(B, p))[4]) // <- picks out the fourth matrix

/*
Now iteratively build the 5x5 matrix M
*/

m := IdentityMatrix(K, 5);
m[5,1] := -t^2;
B := B^m; g5 := m*g5;

m := IdentityMatrix(K, 5);
m[4,5] := 1;
B := B^m; g5 := m*g5;

m := IdentityMatrix(K, 5);
m[3,5] := -1;
B := B^m; g5 := m*g5;

m := IdentityMatrix(K, 5);
m[3,1] := 4*t^2;
B := B^m; g5 := m*g5;

m := IdentityMatrix(K, 5);
m[2,1] := 1;
B := B^m; g5 := m*g5;

// Now we are aiming for type (3,4,4)
m := IdentityMatrix(K, 5);
m[3,2] := -3*t^2;
m[4,2] := 2*t^2;
B := B^m; g5 := m*g5;

// rearrange the columns while preserving the 
// grading
m := PermutationMatrix(K, [1,2,5,4,3]);
B := B^m; g5 := m*g5;

PrintSyz(FirstSyzygies(B));

A_1 =                              A_2 =
[ 0 0  0     0              0 ]    [ 0 0 1      1     16 ]
[   0 16    16              1 ]    [   0 1      2     16 ]
[      0 2*t^2 13*s*t + 2*t^2 ]    [     0 15*t^2 16*t^2 ]
[            0            s^2 ]    [            0      0 ]
[                           0 ]    [                   0 ]

A_3 =                              A_4 =
[ 0 0 0        16   0 ]            [ 0 0 1   0      0 ]
[   0 0        16  16 ]            [   0 0   0      0 ]
[     0 s^2 + t^2   0 ]            [     0 s^2 13*s*t ]
[               0 s^2 ]            [         0  2*s^2 ]
[                   0 ]            [                0 ]



In [22]:
// Desingularisation operation
g4 := DiagonalMatrix(K, [p, p, p, 1]);
g5 := DiagonalMatrix(K, [1, 1, 1, 1/p, 1/p]);

new_A := B^<g5,g4>;

In [23]:
assert not IsDivisibleBy(Discriminant(new_A), s);

In [24]:
// Reorder so that e_1 <= ... <= e_4 and 
// f_1 <= ... <= f_5
g4 := PermutationMatrix(K, [4,2,3,1]);
g5 := PermutationMatrix(K, [1,5,2,3,4]);

new_A := new_A^<g5,g4>;

PrintSyz(FirstSyzygies(new_A));

A_1 =                    A_2 =
[ 0 0 0   1  0 ]         [ 0 16 0   s      1 ]
[   0 0 4*t 15 ]         [    0 1 t^2      0 ]
[     0   0  0 ]         [      0   s      2 ]
[         0  s ]         [          0 15*t^2 ]
[            0 ]         [                 0 ]

A_3 =                    A_4 =
[ 0 0 0 0        16 ]    [ 0 0  0              0     0 ]
[   0 1 0      16*s ]    [   0 16 4*s*t + 15*t^2  16*s ]
[     0 0        16 ]    [      0           16*s    16 ]
[       0 s^2 + t^2 ]    [                     0 2*t^2 ]
[                 0 ]    [                           0 ]



In [25]:
print "The new curve is non-singular in the fibre above s=0 \n" cat 
  "since s does not divide the discriminant\n\n";

printf "The desingularised curve has equations\n"
  cat "%o\n\ninside ℙ(O(1) ⊕ O(2)^3)", 
  DefiningPolynomials(UnderlyingScheme(new_A));

The new curve is non-singular in the fibre above s=0 
since s does not divide the discriminant


The desingularised curve has equations
[
16*s^2*X[2]*X[4] + s^2*X[3]^2 + s^2*X[4]^2 + 9*s*t*X[2]*X[4] + 4*s*t*X[3]*X[4] + 4*s*t*X[4]^2 + 
16*s*X[1]*X[2] + s*X[1]*X[3] + s*X[1]*X[4] + 13*t^2*X[2]^2 + 9*t^2*X[2]*X[4] + t^2*X[3]^2 + 
16*t^2*X[3]*X[4] + 13*t^2*X[4]^2 + 9*t*X[1]*X[2] + 4*t*X[1]*X[3] + 4*t*X[1]*X[4],
s*X[2]^2 + 16*s*X[3]*X[4] + 2*X[1]*X[2] + 16*X[1]*X[3] + 16*X[1]*X[4],
s^2*X[2]*X[4] + 4*s*t*X[2]*X[4] + 13*s*t*X[3]*X[4] + s*X[1]*X[2] + s*X[1]*X[3] + s*X[1]*X[4] + 
3*t^2*X[2]^2 + 15*t^2*X[2]*X[3] + 13*t^2*X[2]*X[4] + 2*t^2*X[3]*X[4] + 4*t*X[1]*X[2] + 
13*t*X[1]*X[3] + 2*X[1]^2,
X[2]^2 + 16*X[2]*X[3] + X[3]^2 + 16*X[3]*X[4],
s*X[2]*X[3] + X[1]*X[2] + X[1]*X[3] + 16*X[1]*X[4]
]

inside ℙ(O(1) ⊕ O(2)^3)


## Applications

### Tchirnhausen realisation in degree 5

> **Theorem**: (F.-Vemulapalli) There exists an (explicit) finite union of polytopes $\mathcal{P} \subset \mathbb{R}^4 \times \mathbb{R}^5$ such that $(\mathbf{e}, \mathbf{f}) \in \mathcal{P} \cap (\mathbb{Z}^4 \times \mathbb{Z}^5)$ if and only if there exists a **smooth** degree 5 cover $C \to \mathbb{P}^1$ with
> $$ \deg(a_{ij}^{(k)}) = d_{ij}^{(k)} = f_i + f_j + e_k - (g + 4)$$

### Real multiplication by $\mathbb{Q}(\sqrt{13})$ and level 2 structure

Recall there exists a "Hilbert modular surface" $Y_{13}$ which "parametrises genus 2 curves with real multiplication"
$$Y_{13}(k) \leftrightarrow \{ (C, \iota) \}$$ 
where $C/k$ has genus 2 and $\iota \colon \mathbb{Z}[\tfrac{1 + \sqrt{13}}{2}] \hookrightarrow \mathrm{End}_k(\mathrm{Jac}(C))$ $\qquad$ (+ condition on polarisation)

And one can ask for $\mathrm{Jac}(C)[2](k) = (\mathbb{Z}/2\mathbb{Z})^4$ in which case we get
$$ Y_{13}[2](k) \leftrightarrow \{(C, \iota, \phi) \} $$
where $\phi \colon (\mathbb{Z}/2\mathbb{Z})^4 \cong \mathrm{Jac}(C)[2](k)$ $\qquad$ (+ condition on polarisation)

> **Theorem:** (F.) The surface $Y_{13}[2]$ is birational over $\mathbb{Q}$ to the surface
> $$
  \xi(u,v;w) = 0 \qquad \text{and} \qquad z^2 = \mathrm{disc}_w \; \xi(u,v;w)  $$
> in $\mathbb{A}^4$, where
> \begin{align*}
  \xi(u,v; w) =\,& (u + v + 1) w^3 - (v + 1) (u + v + 1) w^2 \\
  &\hspace{1cm}- (u + v + 1) (uv + u - v) w + u (v + 1) (uv + u + v^2 + 1).
\end{align*}
> The generic genus $2$ curve can be written down on A4 paper.

> **Theorem**: (Elkies--Kumar) There is an isomorphism of function fields $\mathbb{Q}(Y_{13}) \cong K = \mathbb{Q}(g,h)$ together with explicit (invariants of the) generic genus $2$ curve $\mathcal{C}/\mathbb{Q}(Y_{13})$.

> **Theorem**: (Gruenewald) The field $K(\mathrm{Jac}(\mathcal{C}))[2]$ is isomorphic to the splitting field of an explicit degree $5$ polynomial $f(x) \in K[x]$.

In [26]:
// Gruenewald's polynomial
K<g,h> := FunctionField(Rationals(), 2);
P<x> := PolynomialRing(K);

f := x^5 + (-4*g + 2)*x^4 + (-8*g^2 + 288*g*h + 8*g - 2)*x^3 + (64*g^3 + 
    432*g^2*h^2 - 432*g^2*h - 96*g^2 + 48*g - 8)*x^2 + (-112*g^4 - 
    1728*g^3*h^2 - 1728*g^3*h + 224*g^3 + 21600*g^2*h^2 + 2592*g^2*h - 
    168*g^2 - 864*g*h + 56*g - 7)*x + 64*g^5 - 13824*g^4*h^2 + 2880*g^4*h - 
    160*g^4 + 62208*g^3*h^3 + 19008*g^3*h^2 - 5184*g^3*h + 160*g^3 - 
    41040*g^2*h^2 + 3024*g^2*h - 80*g^2 - 576*g*h + 20*g - 2;

// This cuts out a surface, but I can't spot how to parametrise it...

> **Idea of proof**: Start with Gruenewald's polynomial $f(x) \in K[x]$ and apply the minimisation algorithm (now with base $\mathbb{P}_{\mathbb{Z}}^2\;$). Then the splitting field of $f(x)$ is isomorphic to the splitting field of
> $$ 2x^5 - (2g - 1)x^4 + 2gh x^3 + 2ghx^2 + g^2h^2 $$
> Now parametrise this surface in $\mathbb{A}^3$ (coordinates $(g,h,x)$)... repeat with quartic factor...

In [27]:
f_new := 2*x^5 + (-2*g + 1)*x^4 + 2*g*h*x^3 + 2*g*h*x^2 + g^2*h^2;

L := quo<P | f_new>;
assert #Roots(f, L) eq 1;

In [28]:
print "Thus Gruenewald's polynomial f has a root in ℚ[x]/f_new(x)" 

Thus Gruenewald's polynomial f has a root in ℚ[x]/f_new(x)


In [29]:
// This is how you would go about parametrising the above surface
F<x> := FunctionField(Rationals());
A2_F<g,h> := AffineSpace(F, 2);
eqn := 2*x^5 + (-2*g + 1)*x^4 + 2*g*h*x^3 + 2*g*h*x^2 + g^2*h^2;
C := Curve(A2_F, eqn);
assert Genus(C) eq 0;

## Bonus theorem

By apply an analagous algorithm to the resolvent sextic algebra of the quintic polynomial 
$$ 2x^5 - (2g - 1)x^4 + 2gh x^3 + 2ghx^2 + g^2h^2 $$
one can parametrise the surface parametrising genus $2$ curves with RM by $\mathbb{Q}(\sqrt{13})$ and a rational Weierstrass point. 

> **Theorem**: (F.) The generic genus 2 curve with a rational Weierstrass point and RM by $\mathbb{Q}(\sqrt{13})$ is:

In [30]:
KK<a,b> := FunctionField(Rationals(), 2);
P<x> := PolynomialRing(KK);

In [31]:
c0 := 2*(b^2 + 2*b + 2);
c1 := a^3 + (-2*b^2 + 5*b + 8)*a^2 + (-8*b^3 - 22*b^2 - 10*b - 18)*a + 16*b^4 + 
  20*b^3 + 6*b^2 - 46*b - 48;
c2 := -4*(a^4 + (-2*b^2 + 8*b + 12)*a^3 + (-18*b^3 - 22*b^2 + 28*b + 16)*a^2 + (8*b^5
  + 16*b^4 - 24*b^3 - 12*b^2 - 16*b - 40)*a - 8*b^6 + 8*b^5 + 30*b^4 + 
  40*b^3 - 24*b^2 - 128*b - 92);
c3 := 2*(3*a^5 + (-6*b^2 + 33*b + 48)*a^4 + (-84*b^3 - 50*b^2 + 248*b + 176)*a^3 + 
  (48*b^5 - 52*b^4 - 430*b^3 - 8*b^2 + 496*b + 176)*a^2 + (48*b^6 + 
  272*b^5 - 44*b^4 - 420*b^3 - 232*b^2 - 360*b - 364)*a - 96*b^7 - 112*b^6
  + 152*b^5 + 524*b^4 + 632*b^3 - 256*b^2 - 1076*b - 672);
c4 := -2*(2*a^6 + (-4*b^2 + 28*b + 40)*a^5 + (-76*b^3 - 7*b^2 + 318*b + 230)*a^4 + 
  (48*b^5 - 168*b^4 - 628*b^3 + 264*b^2 + 1256*b + 608)*a^3 + (144*b^6 + 
  304*b^5 - 996*b^4 - 1580*b^3 + 740*b^2 + 1824*b + 624)*a^2 + (544*b^6 + 
  512*b^5 - 1176*b^4 - 1368*b^3 - 208*b^2 - 128*b - 352)*a - 192*b^8 - 
  384*b^7 + 288*b^6 + 1504*b^5 + 1908*b^4 + 144*b^3 - 2964*b^2 - 3704*b - 
  1672);
c5 := (a^3 + (-2*b^2 + 5*b + 8)*a^2 + (-8*b^3 - 2*b^2 + 30*b + 22)*a - 8*b^4 - 
  16*b^3 + 14*b^2 + 42*b + 16)*
  (a^4 + (12*b + 16)*a^3 + (-16*b^3 - 8*b^2 + 52*b + 36)*a^2 + (-16*b^4 - 
  72*b^3 + 24*b^2 + 120*b + 96)*a + 32*b^5 + 16*b^4 - 40*b^3 - 84*b^2 - 
  56*b - 12);
    
f := c0*x^5 + c1*x^4 + c2*x^3 + c3*x^2 + c4*x + c5;